# Cafe Data Analytics Data Cleaning and Transformation

In [ ]:
import pandas as pd
import warnings
import json
import numpy as np
from datetime import datetime, date

In [ ]:
warnings.filterwarnings('ignore')

In [ ]:
pd.set_option('max_colwidth', 2000)

## Load Raw Data from BigQuery

In [ ]:
# Load data from BigQuery into local notebook
df_raw = pd.read_gbq(
    """
        SELECT *
        FROM `jr-data-training.cafe.cafe-sales`
    """,
    project_id='jr-data-training',
    location='australia-southeast1',
)

In [ ]:
# Check the first 5 rows of the raw data
df_raw.head()

In [ ]:
# Check the schema of raw data
print(df_raw.shape)
print(df_raw.info())

The `items` column contains JSON strings of order details. We need to extract fields from this column.

## Data Exploration and Cleaning

In [ ]:
df = df_raw.copy()

### Remove Unsuccessful Order Transactions

In [ ]:
df = df[
    ~pd.isna(df['date_paid'])
]

In [ ]:
df['status'].unique()

Only the status 1 and 2 represent successful transactions, while 10 indicates an unknown status. We'll remove records with status 10 to streamline the dataset.

In [ ]:
df = df[
    df['status'] != 10
].reset_index(drop=True)

In [ ]:
df['status'].unique()

In [ ]:
print(df.shape)
print(df.info())

### Extract field values

In [ ]:
# Try converting the JSON string in 'items' column to dict type
try:
    df['items'] = df['items'].apply(lambda x: json.loads(x))
except Exception as e:
    print(e)

In [ ]:
# View the first items value
df['items'][0]

In [ ]:
# Extract preliminary fields from the 'items' column
for field_name in df['items'][0].keys():
    df[field_name] = df['items'].str[field_name]

In [ ]:
df.columns

In [ ]:
# The extracted 'cart' column contains lists of ordered items
df.loc[0, 'cart']

We need to break down the lists in the `cart` column so that each purchased item in an order is placed in its own row.

In [ ]:
df['cart'].apply(lambda x: type(x)).value_counts()

The `cart` column contains not only list values but also dict values.

In [ ]:
# Sample a 'cart' value that is in dict type
df['cart'].apply(
    lambda x: x if type(x) == dict else None
).value_counts().index[0]

In [ ]:
def convert_to_list(x):
    if type(x) == dict:
        return list(x.values())
    else:
        return x
    
df['cart'] = df['cart'].apply(convert_to_list)

In [ ]:
df['cart'].apply(lambda x: type(x)).value_counts()

In [ ]:
# Separate elements in the `cart` array into multiple rows
df = df.explode('cart', ignore_index=True)
df[['order_id', 'cart']].head()

In [ ]:
# Extract 'item', 'quantity', 'category', and 'price' fields from the 'cart' column
df['item'] = df['cart'].str['name']
df['quantity'] = df['cart'].apply(lambda x: x['quantity'] if 'quantity' in x else 1)
df['category'] = df['cart'].str['category']
df['price'] = df['cart'].str['price']

In [ ]:
# Check if the fields were extracted correctly
df[['order_id', 'item', 'category', 'quantity', 'price']].head()

In [ ]:
# Print the unique categories of sold items
df['category'].unique()

In [ ]:
# Extract options details from 'cart' column
df['options'] = df['cart'].str['options']

In [ ]:
# Convert options format
def extract_options_list(ops):
    if type(ops) == list:
        options_list = []
        for op in ops:
            options_list.append(
                [
                    op['name'],
                    op['value'],
                    op['price'],
                ]
            )
        return options_list
    else:
        return ops

df['options'] = df['options'].apply(extract_options_list)

In [ ]:
df[['order_id', 'item', 'options', 'quantity', 'price']].head()

In [ ]:
# Add 'item_tracking_id' to track the number of items in each order
df['item_tracking_id'] = (
    df.sort_values(['order_id','item'], ascending=[True, True])
      .groupby(['order_id'])
      .cumcount() + 1
)

In [ ]:
df[['order_id', 'item', 'item_tracking_id', 'options']].head(10)

In [ ]:
# Separate elements in the `options` list into multiple rows
df = df.explode('options', ignore_index=True)

In [ ]:
df[['order_id', 'item', 'options', 'quantity', 'price']].head()

In [ ]:
# Extract option's name, value and price from 'options' column
df['option_name'] = df['options'].apply(lambda x: x[0] if type(x) == list else x)
df['option_value'] = df['options'].apply(lambda x: x[1] if type(x) == list else x)
df['option_price'] = df['options'].apply(lambda x: x[2] if type(x) == list else x)

In [ ]:
df[[
    'order_id', 'item', 'options', 'option_name', 'option_value', 'option_price'
]].head()

In [ ]:
# Extract 'size' column
df['size'] = df.apply(
    lambda x: x['option_value']
    if x['option_name'] == 'size'
    else None,
    axis=1,
)

In [ ]:
# Extract 'unit_price' column
df['unit_price'] = df.apply(
    lambda x: x['option_price']
    if x['option_name'] == 'size'
    else None,
    axis=1,
)

In [ ]:
# Create a new dataframe that contains size and unit_price values for each item per order
df_unitprice = (
    df[
        (~pd.isna(df['size'])) & (~pd.isna(df['unit_price']))
      ][
        ['order_id', 'item_tracking_id', 'size', 'unit_price']
    ]
)

df_unitprice.head()

In [ ]:
# Drop the rows that contain 'size' for 'option_name'
# in the original dataframe, and merge the two dataframes together 
df = df[df['option_name'] != 'size'].drop(
    ['size', 'unit_price'], axis=1
).merge(
    df_unitprice,
    how='left',
    on=['order_id', 'item_tracking_id'],
)

In [ ]:
df.columns

In [ ]:
df[
    [
        'order_id', 'item_tracking_id', 'item', 'option_name', 
        'option_value', 'option_price', 'size', 'unit_price'
    ]
].head()

### Drop Irrelevant Columns

In [ ]:
# Remove the columns containing duplicate or redundant information
df.drop(
    ['items', 'cart', 'options'], 
    axis=1, 
    inplace=True,
)

In [ ]:
# Check the data type and unique values of each column
col_details = []
for col in df.columns:
    col_details.append(
        [
            col, 
            df[col].dtype,
            df[col].nunique(), 
            df[col].unique()[:10],
        ],
    )
    
pd.DataFrame(
    col_details,
    columns=[
        'Column Name', 
        'Data Type', 
        'Number of Unique Values', 
        'Unique Value Examples',
    ],
)

In [ ]:
df[
    ['date_created', 'date_paid', 'order_time', 'order_time_verbose']
].drop_duplicates()[:5]

Let's convert `order_time` column to *datetime* format and replace `ASAP` values with the corresponding `date_paid` values.

In [ ]:
# Create a temporary 'order_time_' column that converts 'order_time'
# to datetime format and replace 'ASAP' with corresponding 'date_paid'
df['order_time_'] = df.apply(
    lambda x: datetime.fromtimestamp(int(x['order_time']))
    if x['order_time'] != 'ASAP' 
    else x['date_paid'],
    axis=1,
)

In [ ]:
df[
    ['date_created', 'date_paid', 'order_time', 
     'order_time_', 'order_time_verbose']
].drop_duplicates()[:5]

The other time columns including `date_created`, `date_paid`, `order_time`, `order_time_verbose` are redundant and should be dumped.

In [ ]:
# Drop the redundant timeseries columns
df.drop(
    [
        'date_created',
        'date_paid',
        'order_time',
        'order_time_verbose',
    ],
    axis=1,
    inplace=True,
)

In [ ]:
# Rename 'order_time_' to 'order_time'
df.rename(
    columns={'order_time_': 'order_time'}, inplace=True
)

In [ ]:
# Investigate the '_display' columns and their corresponding numeric columns
df[
    ['cart_gst', 'cart_gst_display',
     'cart_surcharge', 'cart_surcharge_display',
     'cart_total_price', 'cart_total_price_display']
].drop_duplicates().head()

The `cart_gst_display`, `cart_surcharge_display`, and `cart_total_price_display` columns represent the string-formatted prices of the corresponding numeric values in the `cart_gst`, `cart_surcharge`, `cart_total_price` columns. Therefore, those `_display` columns should be removed, and the numeric values should be divided by 100 to accurately reflect the true prices.

In [ ]:
df.drop(
    [
        'cart_surcharge_display', 
        'cart_total_price_display', 
        'cart_gst_display',
    ],
    axis=1, 
    inplace=True,
)

In [ ]:
df.select_dtypes(include='number').drop_duplicates().head()

Divide the quantitative pricing columns, including `total`, `cart_surcharge`, `cart_total_price`, `cart_gst`, `price`, and `option_price`, by 100 to accurately reflect the true prices.

In [ ]:
cols_to_adjust = [
    'total', 'cart_surcharge', 'cart_total_price', 
    'cart_gst', 'price', 'option_price', 'unit_price'
]

for col in cols_to_adjust:
    df[col] = df[col] / 100

In [ ]:
df[cols_to_adjust].drop_duplicates().head()

Since the `cart_size` and `cart_gst` columns can be derived through aggregation, and `total` is the same as `cart_total_price`, we can safely remove the three fields from the dataframe to reduce redundancy.

In [ ]:
# Drop 'cart_size', 'cart_gst' and 'total'
df.drop(
    [
        'cart_size',
        'cart_gst',
        'total',
    ], 
    axis=1, 
    inplace=True,
)

In [ ]:
# Rename 'cart_total_price' and 'price' appropriately
df.rename(
    columns={'cart_total_price': 'order_price', 'price': 'item_price'},
    inplace=True,
)

The fields like `ip_addr`, `order_phone`, `order_name`, and `status` have little relevance to the analysis and can be removed as well to streamline the dataset.

In [ ]:
df.drop(
    [
        'ip_addr', 
        'order_phone',
        'order_name',
        'status',
    ], 
    axis=1, 
    inplace=True,
)

In [ ]:
df.columns

### Impute NULL values

In [ ]:
# Identify the columns with missing values
cols_with_null = []
for col in df.columns:
    if any(pd.isna(x) for x in df[col]):
        cols_with_null.append(col)

In [ ]:
cols_with_null

#### Fill in the missing values for the `category` field

In [ ]:
# Print the unique values of category
df['category'].unique()

In [ ]:
# Find the items that have multiple categories
df_ = (
    df[['item', 'category']][~pd.isna(df['category'])]
    .drop_duplicates().groupby('item').count()
)
items_multiple_cat = list(df_[df_['category'] > 1].index)
items_multiple_cat

In [ ]:
# Check what categories the target items have been assigned to
df[
    (df['item'].isin(items_multiple_cat)) & (~pd.isna(df['category']))
][['item', 'category']].drop_duplicates().sort_values('item')

Let's resolve the conflicts using the most common category for each target item.

In [ ]:
for item in items_multiple_cat:
    most_common_cat = df[df['item'] == item]['category'].value_counts().index[0]
    df.loc[
        df[df['item'] == item].index, 'category'
    ] = most_common_cat

In [ ]:
# Check if the conflicts have been resolved
df[
    (df['item'].isin(items_multiple_cat)) & (~pd.isna(df['category']))
][['item', 'category']].drop_duplicates().sort_values('item')

In [ ]:
# Print the unique values of category
df['category'].unique()

Now the sold items are only categorised into `Hot Drinks`, `Kitchen` and `Cold Drinks`.

In [ ]:
# Find the list of items where the 'category' field is missing
items_missing_cat = list(sorted(df[pd.isna(df['category'])]['item'].unique()))
items_missing_cat

In [ ]:
# Create an dictionary with items as keys and categories as values
df_item_cat = (
    df[~pd.isna(df['category'])]
    [['item', 'category']].drop_duplicates().sort_values('item')
)
dict_item_cat = {x[0]: x[1] for x in df_item_cat.to_numpy()}
dict_item_cat

In [ ]:
# Find the items that are still not classified
items_not_classified = [
    item for item in items_missing_cat if item not in dict_item_cat
]
items_not_classified

In [ ]:
# View the option_name and option_value associated with the unclassified items
df[df['item'].isin(items_not_classified)][
    ['item', 'option_name', 'option_value']
].sort_values('item')

Since `Toasties` and `Toastie` refer to the same item, they should both be classified under `Kitchen`. Given that the `Short Black` item shares similar options with other `Hot Drinks` items, it should be classified under `Hot Drinks`.

In [ ]:
df['item'] = df['item'].apply(
    lambda x: 'Toastie' if x == 'Toasties' else x
)

In [ ]:
# Reflect the re-categorisation of 'Short Black' in dict_item_cat
dict_item_cat['Short Black'] = 'Hot Drinks'

In [ ]:
def impute_category(x):
    if pd.isna(x['category']):
        return dict_item_cat[x['item']]
    else:
        return x['category']
    
df['category'] = df.apply(impute_category, axis=1)

In [ ]:
df['category'].unique()

Now all items have been categorised.

Before we proceed to impute the `cart_surcharge` column, let's take a moment to review the items classified as `Hot Drinks`, `Cold Drinks`, and `Kitchen`.

In [ ]:
pd.DataFrame(
    {
        'category': df['category'].unique(),
        'item': [
            df[df['category'] == cat]['item'].unique() 
            for cat in df['category'].unique()
        ],
    }
)

The `Muffin` item is currently classified as a `Cold Drink`, which seems not sensible...

In [ ]:
# Dive in to investigate the 'Muffin' item
df[df['item'] == 'Muffin'][
    ['item', 'category', 'option_name', 'option_value', 'size', 'unit_price']
].drop_duplicates()

The `Muffin` item has been misclassified. We should be reclassify it under `Kitchen`.

In [ ]:
# Reclassify 'Muffin' items under 'Kitchen'
df.loc[
    df[df['item'] == 'Muffin'].index, 'category'
] = 'Kitchen'

In [ ]:
df[df['item'] == 'Muffin']['category'].unique()

#### Fill in the missing values for the `cart_surcharge` field

In [ ]:
# Check if the 'cart_surcharge' field has missing values
any(pd.isna(v) for v in df['cart_surcharge'].unique())

In [ ]:
# Let's evaluate the relationship between 'cart_surcharge' and 'order_price'
df_ = df[['cart_surcharge', 'order_price']][~pd.isna(df['cart_surcharge'])].copy()
df_['factor'] = df_['cart_surcharge'] / df_['order_price']

In [ ]:
df_['factor'].value_counts()

In [ ]:
df_['factor'].value_counts()[0] / df_['factor'].value_counts().sum()

Let's apply the most frequent factor, 0 (99%), to calculate the unkown `cart_surcharge`.

In [ ]:
# Set the missing value in 'cart_surcharge' column to 0 
df['cart_surcharge'] = df['cart_surcharge'].apply(
    lambda x: 0 if pd.isna(x) else x
)

In [ ]:
# Check if the missing values in 'cart_surcharge' are all filled
any(pd.isna(i) for i in df['cart_surcharge'].unique())

In [ ]:
df.head()

### Calculate Customer Churn Threshold

In [ ]:
# Create a new dataframe for labeling churns
df_churn = df[
    ['customer_id', 'order_time', 'order_id']
].drop_duplicates().sort_values(
    ['customer_id', 'order_time', 'order_id'],
    ascending=[True, True, True],
).reset_index(drop=True).copy()

In [ ]:
# Rename 'order_time' to 'prev_order_time'
df_churn.rename(
    columns={'order_time': 'prev_order_time'},
    inplace=True
)

df_churn.head()

In [ ]:
# Create a new column 'next_order_time' 
# for calculating purchase intervals
df_churn['next_order_time'] = df_churn.sort_values(
    by=['customer_id', 'prev_order_time'], 
    ascending=[True, True],
).groupby(['customer_id'])['prev_order_time'].shift(-1)

df_churn.head()

In [ ]:
# Create new column for recording purchasing interval in days
def get_interval(x):
    if pd.isna(x['next_order_time']):
        return None
    else:
        return (
            x['next_order_time'].date()
            - x['prev_order_time'].date()
        ).days

df_churn['purchase_interval_days'] = (
    df_churn.apply(
        lambda x: get_interval(x), axis=1
    )
)

df_churn.head()

In [ ]:
# Calculate the standard deviation of purchasing 
# interval days for each customer
df_churn['interval_std'] = (
    df_churn.groupby('customer_id')
    ['purchase_interval_days'].transform(np.std)
)

df_churn.head()

In [ ]:
# Calculate the churn threshold as two times the standard 
# deviation of the intervals per customer
df_churn['churn_threshold'] = df_churn['interval_std'] * 2
df_churn.head()

In [ ]:
# Create a new dataframe that maps the churn threshold 
# to each customer
df_customer_churn = df_churn[
    ['customer_id', 'churn_threshold']
].drop_duplicates().reset_index(drop=True)

print(df_customer_churn.shape)
print(df_customer_churn.info())
df_customer_churn.head(10)

For customers with NaN values in the `churn_threshold`, it likely indicates they have only placed one order. In this case, their churn threshold can be set as twice the standard deviation of purchase intervals across all customers.

In [ ]:
churn_threshold_all = np.std(df_churn['purchase_interval_days']) * 2
print(churn_threshold_all)

In [ ]:
# Impute NaN in 'churn_threshold' with two times the standard 
# deviation of the intervals across all customers
df_customer_churn['churn_threshold'] = (
    df_customer_churn['churn_threshold'].apply(
        lambda x: churn_threshold_all if pd.isna(x) else x
    )
)

df_customer_churn.head(10)

In [ ]:
df = df.merge(
    df_customer_churn,
    how='left',
    on='customer_id',
)

In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
print(df.shape)
print(df.info())

### Export the Cleaned Data to a CSV file

In [ ]:
df.to_csv(
    './cafe_analytics_cleaned_CSV_MX/cafe_analytics_full_mx.csv', 
    header=True, 
    index=False,
)

<hr>

## Split the DataFrame Based on the Three Item Categories

In [ ]:
# Split the dataframe based on the 3 categories of sold items
df_hotdrinks = df[df['category'] == 'Hot Drinks'].copy()
df_colddrinks = df[df['category'] == 'Cold Drinks'].copy()
df_kitchen = df[df['category'] == 'Kitchen'].copy()

In [ ]:
# Display the option names and item sizes for each category
# in a dataframe
def display_option_size():
    option_names = [
        df[
            df['category'] == cat
        ]['option_name'].unique()
        for cat in df['category'].unique()
    ]

    sizes = [
        df[
            df['category'] == cat
        ]['size'].unique()
        for cat in df['category'].unique()
    ]

    return pd.DataFrame(
        {
            'Item Category': df['category'].unique(),
            'Option Names': option_names,
            'Size': sizes,
        }
    )

display_option_size()

* The `Size` for **Kitchen** items differs from that of Hot Drinks and Cold Drinks items. To facilitate future product analysis, we can append the `Size` values to the respective item names.
* The `Extra` and `Extras` options for **Kitchen** seem identical, so we should rename them to a consistent name to prevent confusion.
* Similarly, the `Flavour` option values for **Cold Drinks** items can also be appended to item names to further support product analysis.

We will implement those transformations in the following sections.

### Transform Hot Drinks Data for Better Usability

In [ ]:
df_hotdrinks.head()

#### Pivot `option_name` into headers

In [ ]:
def pivot_options(df, pivot_value_col):
    # Pivot the 'option_name' column into headers, 
    # using the 'option_value' column for their corresponding values
    df_pivoted = df.pivot(
        index=['order_id', 'item_tracking_id'], 
        columns='option_name', 
        values=pivot_value_col,
    ).reset_index(drop=False)
    
    # Gather other attributes for each item per order
    df_attr = df.drop(
        ['option_name', pivot_value_col, 'option_price'], 
        axis=1,
    ).drop_duplicates()

    # Join the two tables to create a final table,
    # ensuring each row represents one item per order
    df_final = df_attr.merge(
        df_pivoted, how='inner', on=['order_id', 'item_tracking_id']
    )
    return df_final

In [ ]:
df_hotdrinks_final = pivot_options(df_hotdrinks, 'option_value')

In [ ]:
# Check the pivoted table schema
print(df_hotdrinks_final.shape)
print(df_hotdrinks_final.columns)
df_hotdrinks_final.head()

#### Create a new `option_price` column

In [ ]:
def calc_option_price(df):
    # Create a new 'option_price' column that represents 
    # the difference between 'item_price' per unit and 'unit_price'
    df['option_price'] = (
        df['item_price'] / df['quantity'] - df['unit_price']
    )
    return df

In [ ]:
df_hotdrinks_final = calc_option_price(df_hotdrinks_final)

In [ ]:
# Find the order that includes the highest number of hot drinks
df_hotdrinks_final['order_id'].value_counts()[:1]

In [ ]:
# Check the 'order_price', 'item_price', 'unit_price' 
# and 'option_price' for the sampled order 17135
df_hotdrinks_final[df_hotdrinks_final['order_id'] == 17135][
    [
        'order_id', 'item_tracking_id', 'item', 'size', 'quantity',
        'order_price', 'item_price', 'unit_price', 'option_price'
    ]
]

Now, let's clean the option columns, including `Decaf`, `Equal Sugar`, `Extra shot`, `Honey`, `Milk`, `Raw Sugar`, `Strength`, `Syrup`, `Temp`, and `White Sugar`.

#### Impute missing options

In [ ]:
option_columns = df_hotdrinks['option_name'].unique()
option_columns

In [ ]:
def get_item_option_dict(df):
    items = list(df['item'].unique())

    # Create a dictionary with 'item' as keys and their
    # corresponding 'option_name' as values 
    item_option_dict = {}
    for item in items:
        item_option_dict[item] = (
            df[
                df['item'] == item
            ]['option_name'].unique()
        )
        
    return item_option_dict

In [ ]:
hotdrink_item_option_dict = get_item_option_dict(df_hotdrinks)

In [ ]:
# Check the unique options associated with each item
def show_options_per_item(df, item_option_dict, item_cat):    
    # Check if any applicable option column for each item has NULl values
    items = item_option_dict.keys()
    has_null_list = []
    for item in items:
        has_null = np.any(
            pd.isna(df[item_option_dict[item]][
                df['item'] == item
            ]).to_numpy() == True
        )
        has_null_list.append(has_null)
   
    return pd.DataFrame(
        {
            item_cat: item_option_dict.keys(), 
            'Applicable Options': item_option_dict.values(),
            'Has NULL': has_null_list,
        }
    )

In [ ]:
show_options_per_item(
    df_hotdrinks_final, hotdrink_item_option_dict, 'Hot Drinks'
)

In [ ]:
# Impute the applicable option columns only for each item with the most common choice
def impute_options(df, item_option_dict, option_columns):
    has_null_list = []
    for item in item_option_dict.keys():
        # Find the list of applicable option columns for the current item
        item_options = item_option_dict[item]
        for option in option_columns:
            # If the current option column is applicable for the current item
            if option in item_options:
                # Get the most frequent choice for the option
                most_common = df[
                    df['item'] == item
                ][option].value_counts().index[0]

                # Impute the NaN in the option column with the most frequent choice
                df.loc[
                    df[
                        (df['item'] == item)
                        & (pd.isna(df[option]))
                    ].index, option
                ] = most_common

In [ ]:
impute_options(
    df_hotdrinks_final, hotdrink_item_option_dict, option_columns
)

In [ ]:
show_options_per_item(
    df_hotdrinks_final, hotdrink_item_option_dict, 'Hot Drinks'
)

All missing values in the option columns relevant to each item have now been appropriately filled

In [ ]:
def show_unique_options(df, option_columns):    
    unique_options = []
    for col in option_columns:
        unique_options.append(
            list(df[col].unique())
        )

    return pd.DataFrame(
        {
            'Options': option_columns,
            'Unique Values': unique_options,
        }
    )

In [ ]:
show_unique_options(df_hotdrinks_final, option_columns)

In [ ]:
print(df_hotdrinks_final.shape)
print(df_hotdrinks_final.info())

#### Export the Hot Drinks data as a separate CSV file

In [ ]:
df_hotdrinks_final.to_csv(
    './cafe_analytics_cleaned_CSV_MX/cafe_analytics_hotdrinks_pivoted_mx.csv',
    header=True,
    index=False,
)

### Transform Cold Drinks Data for Better Usability

In [ ]:
df_colddrinks = df[df['category'] == 'Cold Drinks'].copy()

In [ ]:
df_colddrinks.reset_index(drop=True, inplace=True)
print(df_colddrinks.shape)
print(df_colddrinks.info())

In [ ]:
df_colddrinks.head()

In [ ]:
def find_options_with_multiple_values(df):
    # Group the number of option_value by 'order_id',
    # 'item_tracking_id', and 'option_name'
    df_ = df[
        ['order_id', 'item_tracking_id', 'option_name', 'option_value']
    ].groupby(
        ['order_id', 'item_tracking_id', 'option_name']
    ).count().sort_values(
        'option_value', ascending=False
    ).reset_index(drop=False).rename(
        columns={'option_value': 'option_value_count'}
    )
    
    # Find which option(s) has more than one corresponding 'option_value'
    return df_[
        df_['option_value_count'] > 1
    ]['option_name'].unique()

In [ ]:
find_options_with_multiple_values(df_colddrinks)

The `Ingredients` option has more than one corresponding `option_value`. Let's dive in to learn this option...

In [ ]:
df_colddrinks[df_colddrinks['order_id'] == 11329][
    ['order_id', 'item_tracking_id', 'option_name', 'option_value']
]

It is clear to see that some items in each order can have more than one `Ingredient`. We can concatenate these option values into a single string.

#### Concatenate `Ingredient` values for each item into single row

In [ ]:
def concat_options(df):
    # Create a new column 'option_value_new' that concatenates 
    # options for each item per order into a single row
    df['option_value_new'] = df[
        ['order_id', 'item_tracking_id', 'option_name', 'option_value']
    ].groupby(
        ['order_id', 'item_tracking_id', 'option_name']
    ).transform(lambda x: ', '.join(x))

    df = df.drop_duplicates(
        ['order_id', 'item_tracking_id', 'option_name', 'option_value_new']
    )
    return df

In [ ]:
df_colddrinks = concat_options(df_colddrinks)

In [ ]:
# Check if the ingredients have successfully been concatenated
df_colddrinks[df_colddrinks['order_id'].isin([11329, 13470, 33377])][
    ['order_id', 'item_tracking_id', 'option_name', 
     'option_value', 'option_value_new']
]

In [ ]:
# Drop the useless 'option_value' column
df_colddrinks.drop('option_value', axis=1, inplace=True)

#### Pivot `option_name` into headers

In [ ]:
df_colddrinks_final = pivot_options(df_colddrinks, 'option_value_new')

In [ ]:
# Check the pivoted table schema
print(df_colddrinks_final.shape)
print(df_colddrinks_final.columns)
df_colddrinks_final.head()

#### Create a new `option_price` column

In [ ]:
# Create a new 'option_price' column that represents 
# the difference between 'item_price' per unit and 'unit_price'
df_colddrinks_final = calc_option_price(df_colddrinks_final)

In [ ]:
# Find the order that includes the highest number of cold drinks
df_colddrinks_final['order_id'].value_counts()[:1]

In [ ]:
# Check the 'order_price', 'item_price', 'unit_price' 
# and 'option_price' for the sampled order 15001
df_colddrinks_final[df_colddrinks_final['order_id'] == 15001][
    [
        'order_id', 'item_tracking_id', 'item', 'size', 'quantity',
        'order_price', 'item_price', 'unit_price', 'option_price'
    ]
]

Now, let's clean all the option columns.

#### Impute missing options

In [ ]:
option_columns = df_colddrinks['option_name'].unique()
option_columns

In [ ]:
# Create a dictionary with 'item' as keys and their
# corresponding 'option_name' as values
colddrink_item_option_dict = get_item_option_dict(df_colddrinks)

In [ ]:
# Check the unique options associated with each item
show_options_per_item(
    df_colddrinks_final, colddrink_item_option_dict, 'Cold Drinks'
)

In [ ]:
impute_options(
    df_colddrinks_final, colddrink_item_option_dict, option_columns
)

In [ ]:
# Check the unique options associated with each item
show_options_per_item(
    df_colddrinks_final, colddrink_item_option_dict, 'Cold Drinks'
)

All missing values in the option columns relevant to each item have now been appropriately filled

In [ ]:
show_unique_options(df_colddrinks_final, option_columns)

#### Append `Flavour` values to item names

In [ ]:
# Append 'Flavour' option values to Cold Drinks item names
df_colddrinks_final['item'] = df_colddrinks_final.apply(
    lambda x: x['item'] + '-' + x['Flavour']
    if not pd.isna(x['Flavour'])
    else x['item'],
    axis=1,
)

In [ ]:
# Drop the 'Flavour' column
df_colddrinks_final.drop('Flavour', axis=1, inplace=True)

In [ ]:
# Check the unique cold drinks item names
sorted(df_colddrinks_final['item'].unique())

In [ ]:
df_colddrinks_final.columns

In [ ]:
print(df_colddrinks_final.shape)
print(df_colddrinks_final.info())

#### Export the Cold Drinks data as a separate CSV file

In [ ]:
df_colddrinks_final.to_csv(
    './cafe_analytics_cleaned_CSV_MX/cafe_analytics_colddrinks_pivoted_mx.csv',
    header=True,
    index=False,
)

### Transform Kitchen Data for Better Usability

In [ ]:
df_kitchen = df[df['category'] == 'Kitchen'].copy()

In [ ]:
df_kitchen.reset_index(drop=True, inplace=True)
print(df_kitchen.shape)
print(df_kitchen.info())

In [ ]:
# Check the unique options for Kitchen items
option_columns = df_kitchen['option_name'].unique()
option_columns

#### Rename the `Extra` option to `Extras`

The `Extras` and `Extra` options appear to be the same. Let's investigate further to identify which items have the `Extra` / `Extras` option.

In [ ]:
# Identify items that have the 'Extra' option
df_kitchen[df_kitchen['option_name'] == 'Extra']['item'].unique()

In [ ]:
# Identify items that have the 'Extras' option
df_kitchen[df_kitchen['option_name'] == 'Extras']['item'].unique()

Since only one item has the `Extra` option, we can rename it to `Extras` to avoid confusion.

In [ ]:
# Rename 'Extra' option to 'Extras'
df_kitchen['option_name'] = df_kitchen['option_name'].apply(
    lambda x: 'Extras' if x == 'Extra' else x
)

In [ ]:
# Re-check the unique options for Kitchen items
df_kitchen['option_name'].unique()

In [ ]:
# identify options in the kitchen dataset that have 
# multiple corresponding values for a single item
find_options_with_multiple_values(df_kitchen)

The `Extras`, `Toppings`, and `Options` option may have multiple corresponding `option_value` for a single item. Let's concatenate these values into a single row.

#### Concatenate `Extras`, `Toppings` and `Options` values for each item into single row

In [ ]:
# Create a new column 'option_value_new' that concatenates ingredients
# together into a single row
df_kitchen = concat_options(df_kitchen)

In [ ]:
# Check if the ingredients have successfully been concatenated
df_kitchen[df_kitchen['order_id'].isin([19889, 15178])][
    ['order_id', 'item_tracking_id', 'option_name', 
     'option_value', 'option_value_new']
].sort_values(['order_id', 'item_tracking_id'], ascending=[True, True])

In [ ]:
# Drop the useless 'option_value' column
df_kitchen.drop('option_value', axis=1, inplace=True)

#### Pivot `option_name` into headers

In [ ]:
df_kitchen_final = pivot_options(df_kitchen, 'option_value_new')

In [ ]:
# Check the pivoted table schema
print(df_kitchen_final.shape)
print(df_kitchen_final.columns)
df_kitchen_final.head()

#### Create a new `option_price` column

In [ ]:
# Create a new 'option_price' column that represents 
# the difference between 'item_price' per unit and 'unit_price'
df_kitchen_final = calc_option_price(df_kitchen_final)

In [ ]:
# Find the order that includes the highest number of kitchen items
df_kitchen_final['order_id'].value_counts()[:1]

In [ ]:
# Check the 'order_price', 'item_price', 'unit_price' 
# and 'option_price' for the sampled order 28760
df_kitchen_final[df_kitchen_final['order_id'] == 28760][
    [
        'order_id', 'item_tracking_id', 'item', 'size', 'quantity',
        'order_price', 'item_price', 'unit_price', 'option_price'
    ]
]

Now, let's clean all the option columns.

#### Impute missing options

In [ ]:
option_columns = list(df_kitchen['option_name'].unique())
option_columns

In [ ]:
# Create a dictionary with 'item' as keys and their
# corresponding 'option_name' as values
kitchen_item_option_dict = get_item_option_dict(df_kitchen)

In [ ]:
# Check the unique options associated with each item
show_options_per_item(
    df_kitchen_final, kitchen_item_option_dict, 'Kitchen Items'
)

In [ ]:
impute_options(
    df_kitchen_final, kitchen_item_option_dict, option_columns
)

In [ ]:
# Check the unique options associated with each item
show_options_per_item(
    df_kitchen_final, kitchen_item_option_dict, 'Kitchen Items'
)

All missing values in the option columns relevant to each item have now been appropriately filled.

In [ ]:
show_unique_options(df_kitchen_final, option_columns)

#### Append `size` values to item names

In [ ]:
# Append 'size' values to Kitchen item names
df_kitchen_final['item'] = df_kitchen_final.apply(
    lambda x: x['item'] + '-' + x['size'],
    axis=1,
)

In [ ]:
# Remove the 'size' column
df_kitchen_final.drop(
    'size', axis=1, inplace=True
)

In [ ]:
sorted(df_kitchen_final['item'].unique())

In [ ]:
print(df_kitchen_final.shape)
print(df_kitchen_final.info())

#### Export the Kitchen data as a separate CSV file

In [ ]:
df_kitchen_final.to_csv(
    './cafe_analytics_cleaned_CSV_MX/cafe_analytics_kitchen_pivoted_mx.csv',
    header=True,
    index=False,
)

### Union all Pivoted Dataframes into a Single One

In [ ]:
df_pivoted = pd.concat(
    [
        df_hotdrinks_final,
        df_colddrinks_final,
        df_kitchen_final,
    ],
    axis=0,
    ignore_index=True,
)

In [ ]:
# Check the unique number of products
df_pivoted['item'].nunique()

In [ ]:
print(df_pivoted.shape)
print(df_pivoted.info())

In [ ]:
df_pivoted.to_csv(
    './cafe_analytics_cleaned_CSV_MX/cafe_analytics_full_pivoted_mx.csv',
    header=True,
    index=False,
)

<hr>